# Overview of Approaches and Deep Learning Architecture For Visual Odometry

###  Deep Learning VO: main families

**A. Pose regression (end-to-end)**

* **Idea:** CNN (or CNN+RNN) regresses $(\mathbf{R},\mathbf{t}) \in SE(3)$ directly from stacked frames.
* **Architectures:** PoseNet-style CNNs → ConvLSTM/GRU for temporal context.
* **Losses:** supervised $\ell_1/\ell_2$ on translation, geodesic loss on rotation; or **self-supervised photometric** (see §III).
* **Pros:** simple inference; can be fast.
* **Cons:** scale drift (monocular), weak geometry priors, generalization risk.

**B. Depth+Pose joint learning (self-supervised SfM-style)**

* **Idea:** one net predicts **depth** $D_t$; another predicts **relative pose** $T_{t\rightarrow s}$. Reproject source $\mathbf{I}_s$ into target with $D_t$ and $T$; train by minimizing **photometric/SSIM** reconstruction.
* **Architectures:** U-Net depth backbones; small PoseNet; sometimes **cost volumes** (stereo) or **transformers**.
* **Extras:** auto-masking for non-rigid pixels, **explainability masks**, multi-scale supervision, **edge-aware smoothness** on depth.
* **Pros:** no GT poses needed; geometry-aware; scales well with data.
* **Cons:** moving objects/occlusions need handling; absolute scale ambiguous (mono).

**C. Geometry-aware networks (differentiable optimization inside)**

* **Idea:** embed **PnP/BA/ICP** as differentiable layers (Gauss-Newton blocks, differentiable bundle adjustment, learned Jacobians/weights).
* **Examples vibe:** DeepV2D-like depth-pose iterative refinement, BA-Net-style layers, **DROID-SLAM-like** dense matching + iterative pose/structure updates.
* **Pros:** better inductive bias; stronger generalization; better consistency.
* **Cons:** more complex; heavier training; careful stability engineering.

**D. Flow- or correspondence-driven VO**

* **Idea:** learn dense optical flow or correspondences; then recover pose via differentiable epipolar geometry, or train end-to-end.
* **Pros:** good on dynamic scenes with robust matchers; integrates with cost volumes/transformers.
* **Cons:** scale ambiguity (mono); need rigidity masks or scene flow.


---

### Core training losses (self-supervised mono/stereo)

Let $I_t$ target, $I_s$ source; predict $D_t$ and $T_{t\rightarrow s}$. For pixel $p$ in $I_t$:

1. **Back-project:** $\mathbf{X} = D_t(p)\,K^{-1}\tilde{p}$
2. **Transform:** $\mathbf{X}_s = T_{t\rightarrow s}\,\mathbf{X}$
3. **Project:** $p' \sim K\,\mathbf{X}_s$ → sample $\hat{I}_t(p) = I_s(p')$

**Photometric loss:**
$\mathcal{L}_{pho} = \alpha \frac{1 - \mathrm{SSIM}(I_t,\hat{I}_t)}{2} + (1-\alpha)\|I_t-\hat{I}_t\|_1$

**Depth smoothness (edge-aware):**
$\mathcal{L}_{sm} = \sum |\partial_x D_t| e^{-|\partial_x I_t|} + |\partial_y D_t| e^{-|\partial_y I_t|}$

**Geometry consistency:**

* Epipolar loss: $\tilde{p}'^\top F \tilde{p} \approx 0$ on learned correspondences.
* Cycle/reprojection min over multiple sources to handle occlusions.
* **Scale constraints:** stereo baseline, IMU priors, or learned scale head.

**Rotation loss (geodesic) for supervised/regularized pose:**
$\ell_R(R,\hat{R}) = \|\log(R^\top\hat{R})\|_2$

---

###  Popular architectural building blocks

* **Backbones:** ResNet/EfficientNet/MobileNet, or ViT/convnext-style hybrids.
* **Temporal:** ConvLSTM/GRU; 1D temporal convs; attention over clips.
* **Transformers:** for long-range associations, global cost volumes, memory (e.g., recurrent matching + pose/BA heads).
* **Cost volumes:** stereo/monocular multi-hypothesis depth refinement.
* **Differentiable solvers:** Gauss-Newton layers, differentiable PnP/ICP, learned robust weights (M-estimation).
* **Uncertainty heads:** aleatoric/epistemic to weight residuals and poses.

---

###  Practical training & engineering tips

* **Data curation:** varied motion/illumination; ensure exposure consistency or use brightness augmentation & learned photometric invariance.
* **Non-rigidity handling:** auto-mask moving objects; min-reprojection over multiple sources; per-pixel uncertainty weighting.
* **Scale:** prefer stereo or occasional depth supervision/IMU to anchor scale; else learn a scale head or post-scale with a known height/velocity prior.
* **Drift control:** keyframes + temporal windows; small **differentiable BA** blocks every N frames; pose-graph fine-tuning at segment ends.
* **Initialization:** identity or gyro-seeded initial pose; pyramids and coarse-to-fine warping reduce bad local minima.
* **Numerics:** SE(3) parametrization via **Lie algebra** $\boldsymbol{\xi} \in \mathbb{R}^6$; compose poses with $\exp(\cdot)$ / $\log(\cdot)$; use geodesic rotation losses.
* **Speed:** share encoders for depth/pose; mixed precision; tile-based warping; keep cost volumes shallow.

---

## I. Types of Loss Functions Used in VO

For a VO/SLAM network that regresses pose (or pose + depth), the loss has to live on $SE(3)$ — a non-Euclidean manifold. This section walks through the standard families:

* **1.1** Rotation losses — quaternion L2 (with the double-cover fix), and the **geodesic / rotation-angle loss** on $SO(3)$.
* **1.2** Full $SE(3)$ losses — the weighted-sum form (rotation angle + translation norm) and the true Lie-algebra (`log`) form, with worked numerical examples.

The choice of loss interacts with the regression representation (quaternion, axis-angle, $\mathfrak{se}(3)$ twist) and with how the network is supervised (with GT poses vs. self-supervised photometric reconstruction).

---


### 1.1.1 Rotation Loss Overview

A pose-regression network has to predict a rotation $R \in SO(3)$ — a non-Euclidean object. The loss has to respect that geometry, otherwise it will either misbehave near the parameter boundary (e.g., the quaternion sign ambiguity) or assign inflated gradients to rotations that are actually identical.

The three common choices, covered in §1.1.2–§1.1.4, are:

* **Quaternion L2** — the naive baseline. Simple, but has a *double-cover* problem ($q$ and $-q$ represent the same rotation).
* **Quaternion dot-product** — fixes the double-cover issue by taking $|\langle q_{\text{pred}}, q_{\text{gt}} \rangle|$, but still isn't the true geodesic distance.
* **Geodesic angle on $SO(3)$** — the actual shortest-path distance on the rotation manifold. Always use this for serious work; §1.1.3 is the one to remember.

§1.1.5 closes with practical advice on which to use when.

---


###  1.1.2 Quaternion Loss (Naïve Euclidean Loss)

The simplest approach is to minimize the **L2 distance between quaternions**:

$$
\mathcal{L}_\text{quat} = \| q_\text{pred} - q_\text{gt} \|_2
$$

But there are **two problems**:

1. **Double cover**: $q$ and $-q$ represent the same rotation.
   → If the network predicts $-q_\text{gt}$, the loss will be **large**, even though the rotation is exactly correct.
2. **Euclidean mismatch**: The Euclidean distance between quaternions does not exactly correspond to the **geodesic distance** (shortest path on SO(3)).

**Fix for double cover:**

Take the shorter path:

$$
\mathcal{L}_\text{quat} = 1 - |\langle q_\text{pred}, q_\text{gt} \rangle|
$$

where $\langle \cdot, \cdot \rangle$ is the quaternion dot product.
This gives a loss proportional to the **cosine of half the rotation angle**.

---


### 1.1.3 Geodesic Loss (Rotation-Angle Loss)

The **geodesic distance** between two rotations $R_1, R_2 \in SO(3)$ is the smallest rotation angle that aligns them.
If $R = R_1^\top R_2$, then:

$$
\theta = \cos^{-1}\!\left(\frac{\text{trace}(R) - 1}{2}\right)
$$



**Sanity check:** if $R_1 \approx R_2$, then $R = R_1^\top R_2 \approx R_1^\top R_1 = I$, so $\mathrm{tr}(R) = 3$, giving $\frac{\mathrm{tr}(R) - 1}{2} = 1$ and $\cos^{-1}(1) = 0$ — zero rotation, as expected.


This is the true **shortest path distance on SO(3)** (a proper Riemannian metric).
Loss is typically defined as:

$$
\mathcal{L}_\text{geo} = \theta = \cos^{-1}\!\left(\frac{\text{trace}(R_1^\top R_2) - 1}{2}\right)
$$

If using quaternions, you can avoid matrices:

$$
\mathcal{L}_\text{geo} = 2 \cos^{-1} \!\big( |\langle q_\text{pred}, q_\text{gt} \rangle| \big)
$$

where again we take the absolute value to fix the double-cover issue.

---


### 1.1.4 Geodesic Loss Numerical Example

**Ground truth rotation**: 90° around z-axis

  $$
  q_\text{gt} = \left[\cos(45°), 0, 0, \sin(45°)\right] = [0.7071, 0, 0, 0.7071]
  $$

**Predicted rotation**: 60° around z-axis

  $$
  q_\text{pred} = \left[\cos(30°), 0, 0, \sin(30°)\right] = [0.8660, 0, 0, 0.5]
  $$

Both are already normalized.

---

**Compute Dot Product**

$$
\langle q_\text{pred}, q_\text{gt} \rangle
= (0.8660)(0.7071) + (0)(0) + (0)(0) + (0.5)(0.7071)
$$

$$
= 0.6124 + 0.3536 = 0.9660
$$

Take absolute value (to handle sign ambiguity):

$$
|\langle q_\text{pred}, q_\text{gt} \rangle| = 0.9660
$$

---

**Compute Geodesic Loss (Angle)**

Geodesic loss (in radians):

$$
\mathcal{L}_\text{geo} = 2 \cos^{-1}(0.9660)
$$

Compute step-by-step:

* $\cos^{-1}(0.9660) ≈ 0.2618 \, \text{rad}$
* Multiply by 2 → $\mathcal{L}_\text{geo} ≈ 0.5236 \, \text{rad}$

Convert to degrees:

$$
0.5236 \, \text{rad} × \frac{180°}{\pi} ≈ 30°
$$

---



###  1.1.5 Practical Advice

* **If you only care about small orientation errors** (e.g., fine-tuning a network near correct pose):
  Quaternion dot-product loss or L2 loss is usually fine (cheaper, smooth gradients).

* **If you care about accurate global orientation** (e.g., SLAM, pose estimation, camera relocalization):
  **Geodesic loss is strongly preferred** because it reflects the real physical difference between two orientations.

## 1.2 Full Transformation Loss $SE(3)$


To make an **SE(3) loss** you typically combine:

1. a **rotation term** that measures distance on SO(3) (your geodesic loss), and
2. a **translation term** that measures distance in $\mathbb{R}^3$.

There are two common (and solid) ways to do this.

---

### 1.2.1 Option A — Simple & Effective (weighted sum)

Use the geodesic **rotation angle** (in radians) plus a norm on translation:

$$
\mathcal{L}_{\text{SE(3)}} \;=\; \lambda_R \,\underbrace{\big(2\arccos(|\langle q_{\text{pred}}, q_{\text{gt}}\rangle|)\big)}_{\text{SO(3) geodesic angle}}
\;+\; \lambda_t \,\underbrace{\|\,t_{\text{pred}}-t_{\text{gt}}\,\|_2}_{\text{meters}}
$$

* $\lambda_R$ and $\lambda_t$ balance **units** (radians vs meters).
  Rules of thumb:

  * If typical translation errors are \~0.05–0.2 m and angular errors are \~2–10°, try $\lambda_R\in[0.5,2.0]$, $\lambda_t\in[1,10]$.
  * Tune so both terms contribute similar magnitude early in training.
* Often use **robust norms** (e.g., Huber) on translation.

This is the **go-to baseline**: simple, stable, and strong in practice.

### 1.2.2 Option B — True SE(3) geodesic via Lie Log (advanced)

Compute the **relative transform** $\Delta T = T_{\text{gt}}^{-1}T_{\text{pred}}$, take the **matrix logarithm** to get a 6-vector $\xi = [\omega, v]\in \mathbb{R}^6$ (rotation/translation in the tangent space), then penalize it:

$$
\xi \;=\; \log(\Delta T) \;=\; 
\begin{bmatrix} \omega \\ v \end{bmatrix},\quad
\mathcal{L} \;=\; \|\; W\,\xi \;\|_2
\quad\text{or}\quad
\mathcal{L} \;=\; \|W_\omega \omega\|_2 + \|W_v v\|_2.
$$

* Here $W$ (or $W_\omega, W_v$) sets the relative weighting/units.
* This treats rotation and translation **on the same manifold footing** and is invariant to **left/right multiplication** (choose consistently).
* Slightly more math and careful numerics (small-angle handling).


This version gives you a **true SE(3) tangent-space error**. Use if you want strict group-theoretic consistency (e.g., in pose-graph optimization or when composition/invariance properties matter).

---

### 1.2.3 Numerical Example SE(3) Option-B


We’ll compute
$\Delta T = T_{\text{gt}}^{-1}T_{\text{pred}}$, then $\xi=\log(\Delta T)=[\omega,\,v]\in\mathbb{R}^6$, and a loss $\|\omega\|_2+\|v\|_2$.

---

**Poses**

* Ground truth: rotation **+90° about z**, translation $t_\text{gt}=[1,\,0,\,0]$
* Prediction: rotation **+60° about z**, translation $t_\text{pred}=[1.2,\,0.1,\,0]$

Rotation matrices:

$$
R_z(\phi)=\begin{bmatrix}\cos\phi & -\sin\phi & 0\\ \sin\phi & \cos\phi & 0\\ 0&0&1\end{bmatrix}
$$

$$
R_\text{gt}=R_z(90^\circ),\quad
R_\text{pred}=R_z(60^\circ)
$$

---

#### Relative transform $\Delta T$

$$
R_{\text{rel}} = R_\text{gt}^\top R_\text{pred}
= \begin{bmatrix}
0.8660254 & 0.5 & 0\\
-0.5 & 0.8660254 & 0\\
0 & 0 & 1
\end{bmatrix}
\quad(\text{a }-30^\circ\text{ rotation about }z)
$$

$$
t_{\text{rel}} = R_\text{gt}^\top (t_\text{pred}-t_\text{gt})
= \begin{bmatrix}0.1\\ -0.2\\ 0\end{bmatrix}
$$

---

#### $\log(\Delta T)\Rightarrow [\omega,\,v]$

**SO(3) log (rotation):**

* $\theta=\arccos\big((\mathrm{tr}(R_{\text{rel}})-1)/2\big)=\arccos(0.8660254)=\;0.5235988$ rad $=30^\circ$
* Axis $=\ -\hat z$, so

$$
\omega = \theta\cdot(-\hat z) = \begin{bmatrix}0\\0\\-0.5235988\end{bmatrix}
$$

**SE(3) translation log:**

$$
V = I + \frac{1-\cos\theta}{\theta^2}[\omega]_\times
      + \frac{\theta-\sin\theta}{\theta^3}[\omega]_\times^2,
\qquad v = V^{-1} t_{\text{rel}}
$$

Numerically (for $\theta=0.5236$ rad):

$$
V \approx
\begin{bmatrix}
0.95492966 & 0.25587263 & 0\\
-0.25587263 & 0.95492966 & 0\\
0 & 0 & 1
\end{bmatrix},
\qquad
v \approx
\begin{bmatrix}
0.1500647\\
-0.1692298\\
0
\end{bmatrix}
$$

So

$$
\xi=\log(\Delta T)=
\big[\,\omega;\,v\,\big]
=
\begin{bmatrix}
0\\
0\\
-0.5235988\\
0.1500647\\
-0.1692298\\
0
\end{bmatrix}.
$$

---

#### Example loss

With $\mathcal L = \|\omega\|_2 + \|v\|_2$:

* $\|\omega\|_2 = 0.5235988$ (30° in radians)
* $\|v\|_2 \approx \sqrt{0.1500647^2 + (-0.1692298)^2} \approx 0.2261817$

$$
\boxed{\mathcal L \approx 0.5236 + 0.2262 = 0.7498}
$$

---


### 1.2.4 How to pick weights (very important)

* Units differ: **radians vs meters**. You must balance them.
* Three common strategies:

  1. **Manual tuning** (start with $\lambda_R=1, \lambda_t\in[1,10]$).
  2. **Normalize by dataset scale** (e.g., divide translation by scene extent).
  3. **Learned homoscedastic uncertainty** (Kendall & Cipolla):

     $$
     \mathcal{L} = \frac{1}{2\sigma_R^2} \, \mathcal{L}_R + \frac{1}{2\sigma_t^2}\, \mathcal{L}_t + \log \sigma_R + \log \sigma_t
     $$

     with $\log \sigma_R, \log \sigma_t$ as learnable scalars.

---

### 1.2.5 Quick recommendations

* Start with **Option A** (weighted sum, Huber on translation). It’s robust and easy to tune.
* If you need **group-consistent** errors (e.g., enforcing trajectory smoothness with relative poses), use **Option B** (Lie log).
* Always monitor **angle (deg)** and **translation (m)** separately as metrics, even if your loss is a combination.




## II. Model Design Variants

A monocular VO network makes three architectural decisions:

1. **Backbone** — what extracts features from each frame? (CNN vs. ViT vs. hybrid)
2. **Temporal aggregation** — how is information shared across frames? (recurrent, window-attention, cost volume, BA)
3. **Pose head** — what actually produces $(R, \mathbf{t})$? (MLP regression, iterative refinement, differentiable geometric solver)

This section walks through the design space, focusing on **ViT-based** choices since that's the focus of this notebook.

---

### 2.1 Pose Regression Heads — CNN-style vs ViT-style

The classic **PoseCNN / SfM-Learner pose head** is a small ConvNet:

* Input: a stack of $K{+}1$ consecutive RGB frames concatenated along the channel axis (e.g., $K=2 \Rightarrow$ 9-channel input).
* A few strided conv layers downsample to a tiny spatial grid.
* Global average pool → MLP → 6 outputs per relative pose (3 translation + 3 axis-angle), repeated $K$ times.

This works for short sequences but has two limitations:

* **Inductive bias is purely local** — kernels can't relate distant patches in one frame, or correspond patches across frames except through pooling.
* **Scaling with sequence length is poor** — a fixed-channel input fixes $K$ at training time.

**ViT-style pose heads** address both:

* Each frame is patchified into tokens ($N$ tokens × $D$ dimensions).
* A learned `[pose]` token is appended per relative pose (cf. BERT's `[CLS]`).
* Self-attention is run jointly over all tokens from all frames in the window.
* The `[pose]` tokens are projected to $(R, \mathbf{t})$ via an MLP.

Trade-off: better global modeling, but $\mathcal{O}((K \cdot N)^2)$ attention cost. Mitigations: local-window attention (Swin), axial / factorized space–time attention (TimeSformer).

---

### 2.2 Representative ViT-based VO Architectures

| Model | Year | Idea |
| --- | --- | --- |
| **TSformer-VO** (Françani & Maximo) | 2024 | Factorized space–time attention on consecutive frames; pose token regresses 6-DoF per pair. |
| **DPVO** — Deep Patch Visual Odometry (Teed & Deng) | 2023 | Sparse-patch transformer features + differentiable bundle adjustment. Learned features, classical optimization. |
| **DROID-SLAM** (Teed & Deng) | 2021 | Dense bipartite frame graph + learned GRU update operator + differentiable BA. Transformer-style updates, not pure ViT. |
| **VGGT** — Visual Geometry Grounded Transformer | 2024 | "All frames in one forward pass" — multi-view transformer predicts cameras + depth jointly, no recurrence or BA. |
| **Fast3R** | 2025 | Same family as VGGT, scales to 1000+ images in a single forward pass. |
| **MASt3R-SLAM** | 2024 | Two-view matching transformer fused with a SLAM front-end (keyframes, loop closure). |

**Common pattern in post-DROID work**: the network outputs **dense correspondences or feature maps**, and a **differentiable geometric layer** (BA, PnP, ICP) recovers pose. The transformer is the *matcher*, not the pose regressor. This combines the inductive bias of geometry with the representational power of attention, and is currently the strongest paradigm on KITTI/EuRoC/TUM.

---

### 2.3 Temporal Modeling

How is information shared across frames?

* **Pair-wise (Δ-pose) prediction** — regress $T_{t \to t+1}$ from two frames. Simple, but accumulates drift; no way to refine an earlier pose given later evidence.
* **Window attention** ($K$ frames at once) — predict all $K{-}1$ relative poses in one shot, with self-attention across all window tokens. Better short-horizon consistency.
* **Recurrent state** (DROID-SLAM, DeepVO) — hidden state carries earlier-frame information forward. Handles arbitrary-length sequences naturally but harder to train.
* **Iterative refinement** (DPVO, DROID-SLAM) — predict an *update* to current poses + depths, run for $N$ iterations. Each iteration is a transformer or GRU; the geometry layer (BA / DBA) closes the loop.
* **Global memory** (VGGT, Fast3R) — all frames attend to all others in one forward pass. Most expressive, most expensive.

The trade-off is **latency** (pair-wise is fastest), **drift** (longer windows / recurrent state / BA help most), and **training stability** (iterative methods need careful loss balancing across iterations).

---

### 2.4 Input and Output Representations

**Input representations**:

| Choice | When to use |
| --- | --- |
| Stacked RGB | Default; cheapest. |
| RGB + optical flow | Small motion + weak features (textureless scenes). |
| RGB + monocular depth | If you have a strong off-the-shelf depth net (Depth Anything, MiDaS). Often a free win. |
| Multi-resolution pyramid | Large motion within the window — helps coarse-to-fine attention. |

**Rotation output representations** — the choice interacts with the loss in §I:

| Representation | Dim | Pros | Cons |
| --- | --- | --- | --- |
| Quaternion $(w, x, y, z)$ | 4 | Compact, smooth | Double-cover; not a Lie algebra |
| Axis-angle $\boldsymbol{\omega} \in \mathbb{R}^3$ | 3 | On the Lie algebra; matches geodesic loss naturally | Singular at $\|\boldsymbol{\omega}\| = \pi$ |
| 3×3 rotation matrix (SVD-projected) | 9 | Smooth, no gimbal | Higher-dim; needs projection step |
| **6D continuous rep** (Zhou et al. 2019) | 6 | **Continuous and smooth everywhere** — recommended for direct regression | Slightly more compute to convert |

For **translation**, regress $\mathbf{t} \in \mathbb{R}^3$ directly. Monocular VO is scale-ambiguous; with photometric supervision (§III) plain $\mathbf{t}$ regression works because scale is fixed by the depth prediction.

---


## III. Unsupervised / Self-Supervised VO

"Self-supervised" here means: **no ground-truth poses, no ground-truth depth** — train only from a monocular video stream by enforcing **photometric consistency** between adjacent frames via differentiable warping.

The supervisory signal: if predicted depth and pose are correct, then a pixel in frame $t$, when re-projected into frame $s = t \pm 1$ via the predicted $D_t$ and $T_{t \to s}$, should land on the same color as the corresponding pixel in $I_s$.

---

### 3.1 The SfM-Learner Paradigm (Zhou et al. 2017)

The seminal recipe — **two cooperating networks** trained jointly:

1. **DepthNet** $f_D: I_t \mapsto D_t$ — per-pixel depth (or inverse depth) for the target frame.
2. **PoseNet** $f_P: (I_t, I_s) \mapsto T_{t \to s}$ — relative pose from target to source frame.

A **view-synthesis loss** glues them together:

1. Back-project a pixel $p$ in $I_t$ to 3D: $\mathbf{X} = D_t(p) \cdot K^{-1} \tilde{p}$.
2. Transform to source frame: $\mathbf{X}_s = T_{t \to s} \mathbf{X}$.
3. Project to source pixel coordinates: $p' \sim K \mathbf{X}_s$.
4. Sample the source image: $\hat{I}_t(p) = I_s(p')$ (bilinear).
5. Penalize the reconstruction: $\mathcal{L} = \rho(I_t, \hat{I}_t)$.

Neither network alone is well-defined — depth without pose is scale-ambiguous, pose without depth is scale-and-shape-ambiguous — but the *combination* is constrained.

---

### 3.2 Photometric Reprojection — The Warping in Detail

Given target $I_t$, source $I_s$, predicted depth $D_t$, predicted pose $T_{t \to s} \in SE(3)$, and intrinsics $K$:

$$
\boxed{\;p' \;\sim\; K\, T_{t \to s}\, D_t(p)\, K^{-1}\, \tilde{p}\;}
$$

where $\tilde{p} = (u, v, 1)^\top$ is the homogeneous pixel and $\sim$ means "up to scale" (divide by the last component).

**Implementation tip**: in PyTorch, this is a `grid_sample` with bilinear interpolation. The Jacobian of `grid_sample` w.r.t. its grid argument provides gradients into both $D_t$ and $T_{t \to s}$ — that's why this works end-to-end.

**Boundary handling**:

* Pixels in $I_t$ that project outside $I_s$ provide no signal — mask them out.
* Pixels with very small or invalid depth produce huge gradient noise — clamp $D_t$ to a reasonable range (e.g., $[0.1, 100]$ m).

**Inverse depth**: most papers predict $1/D$ (disparity) rather than $D$ because (a) it's bounded for distant points, (b) photometric error is roughly linear in inverse depth for small camera baselines.

---

### 3.3 Photometric Loss — SSIM + L1

Plain pixel-wise L1 on RGB is poor: dominated by global illumination changes and saturation, ignoring local structure. The standard fix (Garg et al. 2016; Godard et al. 2017) combines **SSIM** with **L1**:

$$
\mathcal{L}_{pho}(I_t, \hat{I}_t) \;=\; \alpha \cdot \frac{1 - \mathrm{SSIM}(I_t, \hat{I}_t)}{2} \;+\; (1 - \alpha) \cdot \| I_t - \hat{I}_t \|_1, \qquad \alpha \approx 0.85.
$$

**SSIM** compares local luminance, contrast, and structure on small Gaussian-windowed patches:

$$
\mathrm{SSIM}(x, y) = \frac{(2\mu_x \mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}
$$

with stability constants $c_1 = 0.01^2$, $c_2 = 0.03^2$ (for images in $[0, 1]$).

Intuition: SSIM is **invariant to global brightness offsets**, so it tolerates exposure changes between frames — a frequent issue in real video.

---

### 3.4 Two Critical Tricks — Auto-Masking and Min-Reprojection

Naïve photometric loss has two killer failure modes:

1. **Moving objects** (cars, pedestrians) violate the static-scene assumption. The network learns to predict infinite depth on them to "remove" them from the warped image.
2. **Occlusions** at object boundaries: a pixel visible in $I_t$ may be hidden in $I_s$. Forcing reconstruction creates wrong gradients.

**Monodepth2 (Godard et al. 2019)** introduced two fixes that have become standard.

**Auto-masking** — drop pixels where the **un-warped** source already matches the target better than the warped one:

$$
\mu(p) \;=\; \Big[\, \mathcal{L}_{pho}(I_t, \hat{I}_t)(p) \;<\; \mathcal{L}_{pho}(I_t, I_s)(p) \,\Big]
$$

(Iverson bracket; 1 if true.) Catches **objects that move with the camera** (e.g., the car ahead at constant relative velocity, which looks identical in $I_t$ and $I_s$ no matter what the warp does) and **stationary-camera frames** (no parallax → no signal).

**Min-reprojection** — compute the photometric loss against *every* source frame $I_{s_1}, I_{s_2}, \dots$ and take the **per-pixel minimum**:

$$
\mathcal{L}^*_{pho}(p) \;=\; \min_{s \in \mathcal{S}} \mathcal{L}_{pho}(I_t, \hat{I}_t^{(s)})(p).
$$

Rationale: a pixel occluded in one source frame is likely visible in another; the per-pixel min selects the source where it's visible.

Together, these two tricks dropped the KITTI absolute relative depth error from ~0.115 → ~0.106 — a notable jump for what amounts to two lines of code.

---

### 3.5 Auxiliary Losses and Scale Anchoring

**Edge-aware smoothness** — regularize the depth so it varies smoothly *except* across image edges:

$$
\mathcal{L}_{sm} \;=\; \sum_{p} \Big( |\partial_x D^*_t(p)|\, e^{-|\partial_x I_t(p)|} \;+\; |\partial_y D^*_t(p)|\, e^{-|\partial_y I_t(p)|} \Big)
$$

where $D^*_t = D_t / \overline{D_t}$ is the **mean-normalized** inverse depth (prevents the "shrinking depth" failure where the network globally minimizes gradient magnitude by collapsing depth toward zero).

The exponential weight $e^{-|\partial I|}$ relaxes the smoothness penalty across image edges, where real depth discontinuities are expected.

**Multi-scale supervision** — apply the photometric loss at multiple decoder resolutions (typically 4 scales, ×2 downsample each), upsampled to full resolution before computing the loss. Helps when the coarse prediction is far from correct.

**Scale ambiguity** — monocular self-supervised VO is fundamentally scale-ambiguous: a free parameter $s$ makes $(D / s, s \cdot T)$ give the same photometric loss as $(D, T)$. Four ways to anchor it:

| Technique | How it anchors scale |
| --- | --- |
| **Stereo pairs** | Known baseline $b$ gives metric scale via stereo photometric loss. |
| **IMU pre-integration** | Inertial velocity integrates to metric translation (the basis of all VIO). |
| **Velocity / height priors** | Assume known camera height (driving) or known speed (mobile). Loose but useful. |
| **Median scaling at eval** | At test time, scale the prediction so its **median depth** matches GT. Common in monocular KITTI papers — but always report it explicitly. |

Without one of these, your test-time numbers are only meaningful **up to scale** — evaluate with **Sim(3) alignment** (§4.2), not SE(3).

---


## IV. Trajectory Evaluation Metrics

Once a VO/SLAM system produces a trajectory, the next question is **how good is it?** Comparing an estimated trajectory $\{\hat{T}_i\}_{i=1}^N$ to a ground-truth trajectory $\{T_i^{gt}\}_{i=1}^N$ is **not** as simple as taking a per-frame distance: monocular systems have a free **scale**, every VO has a free **gauge** (initial pose), and inertial systems leave **yaw** free. You have to **align** before you measure.

This section follows the recommendations of:

* **Z. Zhang and D. Scaramuzza**, *"A Tutorial on Quantitative Trajectory Evaluation for Visual(-Inertial) Odometry"*, **IROS 2018** ([PDF](https://rpg.ifi.uzh.ch/docs/IROS18_Zhang.pdf))
* **J. Sturm et al.**, *"A Benchmark for the Evaluation of RGB-D SLAM Systems"*, IROS 2012 (the original ATE/RPE definitions used by TUM RGB-D).
* **A. Geiger et al.**, *"Are we ready for autonomous driving? The KITTI vision benchmark suite"*, CVPR 2012 (the sub-trajectory drift style).

and presents both the math and the tools you actually run in practice (`evo`, `rpg_trajectory_evaluation`).

---


### 4.1 Why naive comparison fails — the alignment problem

A VO/SLAM trajectory is only defined **up to an unobservable gauge transform**. Two systems can be equally accurate in the geometry they recover but live in completely different reference frames.

The set of "free" degrees of freedom depends on the **sensor configuration**:

| System                          | Unobservable DoF                  | Required alignment |
| ------------------------------- | --------------------------------- | ------------------ |
| Monocular VO                    | 7 DoF: 6-DoF pose + **scale**     | **Sim(3)**         |
| Stereo / RGB-D / Lidar          | 6 DoF: initial pose               | **SE(3)**          |
| Visual-Inertial (VIO)           | 4 DoF: position + **yaw**         | **posyaw**         |
| Inertial-only / gravity-aligned | 1 DoF: yaw                        | **yaw-only**       |

Aligning over the **correct** set of unobservable DoFs is critical:

* Using a **stronger** alignment than necessary (e.g., Sim(3) for a VIO system) *artificially* hides real scale drift that the system is supposed to be free of — the headline number looks great but the system is broken.
* Using a **weaker** alignment than necessary (e.g., yaw-only on monocular VO) produces huge errors that have nothing to do with the algorithm's accuracy.

**Rule (Zhang & Scaramuzza):** align over exactly the unobservable DoFs of the system, and nothing more.

---


### 4.2 Sim(3) alignment (Umeyama, 1991) — for monocular

For monocular VO, you align with a **similarity transform** $(s, R, \mathbf{t}) \in \mathrm{Sim}(3)$ that minimizes:

$$
(s^\star, R^\star, \mathbf{t}^\star) \;=\; \arg\min_{s, R, \mathbf{t}}\; \sum_{i=1}^{N}\, \big\| \, s\,R\,\hat{\mathbf{p}}_i + \mathbf{t} \;-\; \mathbf{p}_i^{gt}\,\big\|_2^{2}
$$

where $\hat{\mathbf{p}}_i$ are estimated positions and $\mathbf{p}_i^{gt}$ are ground-truth positions. **Umeyama's closed form**:

1. Compute centroids $\bar{\mathbf{p}}, \bar{\mathbf{q}}$ and centered points.
2. Cross-covariance $\Sigma = \frac{1}{N}\sum_i (\mathbf{q}_i - \bar{\mathbf{q}})(\mathbf{p}_i - \bar{\mathbf{p}})^\top$ and SVD $\Sigma = U D V^\top$.
3. Let $S = \mathrm{diag}(1, 1, \det(UV^\top))$ to enforce a proper rotation.

$$
R^\star = U \, S \, V^\top, \qquad
s^\star = \frac{\mathrm{tr}(DS)}{\sigma_p^2}, \qquad
\mathbf{t}^\star = \bar{\mathbf{q}} - s^\star R^\star \bar{\mathbf{p}}
$$

with $\sigma_p^2 = \frac{1}{N}\sum_i \|\mathbf{p}_i - \bar{\mathbf{p}}\|^2$.

The recovered scale $s^\star$ is itself a useful **diagnostic** — log it. A consistent $s^\star \neq 1$ across multiple sequences is a sign of a systematic scale bias in the model.

---


### 4.3 Other alignments — SE(3), yaw-only, posyaw

* **SE(3) (rigid)** — set $s = 1$ in the Sim(3) problem. Closed form is the same as Umeyama with the scale step skipped (a.k.a. *orthogonal Procrustes* / *Kabsch*). Use for stereo, RGB-D, lidar, and anything that produces metric depth.

* **Yaw-only** — align rotation around the gravity direction $\mathbf{g} = \hat{\mathbf{z}}$ only (roll and pitch are observable from gravity in inertial systems). Solve for $\psi \in [-\pi, \pi]$:

$$
\psi^\star = \arg\min_{\psi}\, \sum_i \big\| R_z(\psi)\hat{\mathbf{p}}_i + \mathbf{t} - \mathbf{p}_i^{gt} \big\|_2^2
$$

with closed-form solution by setting the derivative w.r.t. $\psi$ to zero — yields an `atan2` expression on the 2D projection of the cross-covariance.

* **posyaw** — align translation **and** yaw (4 DoF total), but not roll/pitch/scale. This is the recommended alignment for **VIO** evaluation: it leaves gravity-aligned roll/pitch and metric scale free, both of which a VIO is supposed to recover correctly.

---


### 4.4 Absolute Trajectory Error (ATE)

After alignment, for each timestamped pose pair $(T_i^{gt}, \hat{T}_i^{aligned})$, define the absolute error in **SE(3)** as

$$
E_i^{abs} \;=\; (T_i^{gt})^{-1}\,\hat{T}_i^{aligned}.
$$

Splitting $E_i^{abs}$ into rotation $R_i^{err}$ and translation $\mathbf{t}_i^{err}$:

$$
\mathrm{ATE}^{trans}_i = \|\mathbf{t}_i^{err}\|_2 \;\;[\text{m}],\qquad
\mathrm{ATE}^{rot}_i = \big|\angle\, R_i^{err}\big| = \arccos\!\Big(\tfrac{\mathrm{tr}(R_i^{err}) - 1}{2}\Big) \;\;[\text{rad}].
$$

Aggregate statistics:

$$
\mathrm{ATE}^{trans}_{\mathrm{RMSE}} = \sqrt{\tfrac{1}{N}\sum_i \|\mathbf{t}_i^{err}\|^2}, \qquad
\mathrm{ATE}^{trans}_{\mathrm{median}} = \mathrm{median}_i \|\mathbf{t}_i^{err}\|.
$$

**Strengths** — single intuitive number, good for assessing **global consistency** (loop closures, drift bounding).

**Weaknesses** — and this is what Zhang & Scaramuzza specifically warn about:

* **Highly sensitive to where drift happens**: a single large excursion at the end of the trajectory dominates the RMSE.
* **Depends on alignment choice**: switching from SE(3) to Sim(3) can change the number by an order of magnitude on monocular trajectories.
* **Not normalized by path length**: a 1 m ATE on a 10 m sequence is awful; on a 10 km sequence it's excellent. ATE values across different sequences are not directly comparable.

For these reasons, **never report ATE alone** — pair it with sub-trajectory drift (§4.6).

---


### 4.5 Relative Pose Error (RPE)

RPE measures **local consistency** over a fixed window $\Delta$ (in time or in distance). Defined per Sturm et al. (TUM RGB-D):

$$
E_i^{rel}(\Delta) \;=\; \Big((T_i^{gt})^{-1}\, T_{i+\Delta}^{gt}\Big)^{-1} \, \Big(\hat{T}_i^{-1}\, \hat{T}_{i+\Delta}\Big).
$$

That is: take the GT relative motion from frame $i \to i{+}\Delta$, the estimated relative motion from frame $i \to i{+}\Delta$, and measure their mismatch.

Translation/rotation statistics are formed the same way as ATE:

$$
\mathrm{RPE}^{trans}_{\mathrm{RMSE}}(\Delta) = \sqrt{\tfrac{1}{N-\Delta}\sum_i \big\| \mathbf{t}\big(E_i^{rel}(\Delta)\big) \big\|^2}.
$$

**Properties**:

* No global alignment is needed — RPE is **invariant to gauge**.
* For $\Delta = 1$ frame, RPE measures the **inter-frame drift rate**.
* For large $\Delta$, it approaches ATE behavior over that window.

The weakness of plain RPE: the choice of $\Delta$ is somewhat arbitrary. A single $\Delta$ in seconds is hard to compare across sequences with different velocities. The fix is to use **multiple sub-trajectory lengths** — the topic of the next section.

---


### 4.6 Sub-trajectory drift — the recommended evaluation

**This is the central recommendation of Zhang & Scaramuzza (IROS 2018), and also the methodology behind the KITTI benchmark leaderboard.**

Idea: instead of one alignment and one RPE window, evaluate over **many sub-trajectories of different lengths** distributed along the trajectory.

**Algorithm**:

1. Pick a set of segment lengths $\mathcal{L} = \{L_1, L_2, \dots, L_k\}$.
   * KITTI uses $\{100, 200, 300, 400, 500, 600, 700, 800\}$ m.
   * For indoor / drone sequences, scale down (e.g., $\{2, 4, 6, 8, 10, 20\}$ m).
2. For each $L \in \mathcal{L}$, slide along the trajectory and extract every sub-trajectory whose **ground-truth path length** is approximately $L$.
3. **Align each sub-trajectory locally** at its starting pose (this is the key — no global alignment, only at the segment start).
4. Compute the end-to-end translation and rotation error for that segment.
5. **Normalize by length**:

$$
e_t(L) = \frac{\|\mathbf{t}_{end}\|}{L} \;\;[\%\,\text{or m/m}], \qquad
e_R(L) = \frac{|\angle R_{end}|}{L} \;\;[\text{deg/m or rad/m}].
$$

6. Report the **distribution** (median, RMSE, percentiles) of $e_t(L)$ and $e_R(L)$ for each $L$, typically as a boxplot vs. distance.

**Why this is better**:

* **Normalized by path length** → numbers are directly comparable across sequences and datasets.
* **Reveals the scale at which drift occurs**: a system can have low error at 100 m and high error at 800 m (slow drift accumulation). A single ATE number hides this completely.
* **Robust statistics** (median, IQR over many segments) reduce the influence of single bad excursions.

This is exactly what **KITTI's leaderboard** reports as $t_{rel}$ (% translation error) and $r_{rel}$ (deg/100 m).

---


### 4.7 Which statistics to report

Zhang & Scaramuzza recommend reporting **more than just the mean / RMSE**:

| Statistic                  | Why it matters                                                                            |
| -------------------------- | ----------------------------------------------------------------------------------------- |
| **RMSE**                   | Standard, penalizes outliers — but is itself outlier-sensitive.                           |
| **Median**                 | Robust to a small number of bad frames.                                                   |
| **25th / 75th percentile (IQR)** | Shape of the error distribution.                                                    |
| **Max**                    | Worst-case behavior; matters when failures are unacceptable (e.g., autonomous driving).   |

For **sub-trajectory drift**, a **boxplot of error vs. segment length** is the most informative single visualization. For **ATE**, both the **trajectory plot** (XY, optionally XZ) overlaid with ground truth and the **error-vs-time curve** should be shown.

When comparing systems, *always* state:

1. The **alignment used** — Sim(3) / SE(3) / posyaw / yaw-only.
2. The **timestamp matching method** — nearest-neighbor in time, max time delta.
3. **Which frames are excluded** — initialization frames, lost-track regions, the first N frames used only for alignment.
4. Whether the trajectory was **trimmed** to match GT length.

Tiny differences in these conventions cause large differences in the headline number.

---


### 4.8 Tools — `evo` and `rpg_trajectory_evaluation`

In practice you almost never compute these by hand. Three standard toolboxes:

---

#### `evo` (Michael Grupp) — [github.com/MichaelGrupp/evo](https://github.com/MichaelGrupp/evo)

The most popular general-purpose tool; supports KITTI, TUM, EuRoC, ROS bag, and ROS2 formats.

```bash
pip install evo --upgrade

# Absolute Trajectory Error, Sim(3) alignment, KITTI format
evo_ape kitti gt.txt est.txt -va -s --plot --plot_mode xz --save_results ape.zip

# Relative Pose Error with 100 m windows
evo_rpe kitti gt.txt est.txt -va -s --delta 100 --delta_unit m --plot

# Trajectory visualization with ground-truth reference
evo_traj kitti est.txt --ref=gt.txt -p --plot_mode=xyz

# TUM format with associate-by-timestamp
evo_ape tum gt.txt est.txt -va -as --plot      # -a SE(3), -as Sim(3)
```

Key flags:

* `-a` SE(3) alignment, `-as` Sim(3) (with scale), `-ap` posyaw, `--align_yaw` yaw-only.
* `-s` scale-correct (Sim(3) shorthand for `-a -s`).
* `--n_to_align N` use only the first N poses for alignment (the rest are evaluated unaligned).
* `--save_results out.zip` produces a pickleable result you can re-plot with `evo_res out.zip --use_filenames -p`.

---

#### `rpg_trajectory_evaluation` (Zhang & Scaramuzza, MIT) — [github.com/uzh-rpg/rpg_trajectory_evaluation](https://github.com/uzh-rpg/rpg_trajectory_evaluation)

The toolbox that accompanies the **IROS 2018 paper**. Where `evo` is the "Swiss Army knife", this is the **paper-ready batch-evaluation** tool — it's how you'd typically generate the comparison tables and boxplots that appear in VO/SLAM publications.

**Two entry points**:

* `analyze_trajectory_single.py` — single algorithm on a single sequence.
* `analyze_trajectories.py` — **batch mode**: a matrix of algorithms × sequences, with multiple runs per (algo, seq) for variance reporting. Produces:
  * Per-dataset trajectory overlays.
  * **Sub-trajectory drift boxplots** across algorithms (the §4.6 methodology).
  * Per-algorithm RMSE boxplots across datasets.
  * **LaTeX tables** (mean / median / min / max / std) ready to paste into a paper.

**Input file format** (per trajectory):

```text
# stamped_groundtruth.txt  AND  stamped_traj_estimate.txt
# columns: timestamp tx ty tz qx qy qz qw   (space-separated)
0.000000  0.0000  0.0000  0.0000   0.0000  0.0000  0.0000  1.0000
0.050000  0.0123  0.0001 -0.0004   0.0000  0.0000  0.0021  1.0000
...
```

**Directory layout** for batch mode:

```
results/
  <platform>/                           # e.g., laptop, jetson
    <algo>/                             # e.g., orbslam3, vins_fusion
      <platform>_<algo>_<dataset>/
        stamped_groundtruth.txt
        stamped_traj_estimate.txt
        eval_cfg.yaml                   # optional: alignment + segments
        start_end_time.yaml             # optional: temporal crop
```

**`eval_cfg.yaml`** (per-trajectory configuration):

```yaml
align_type: sim3                # sim3 | se3 | posyaw | none
align_num_frames: -1            # -1 = use all, else first N
```

**Typical workflow**:

```bash
git clone https://github.com/uzh-rpg/rpg_trajectory_evaluation
cd rpg_trajectory_evaluation
python scripts/analyze_trajectory_single.py /path/to/results_dir
# or batch:
python scripts/analyze_trajectories.py analyze_trajectories_config/euroc_vio.yaml \
    --results_dir=/path/to/results --output_dir=/path/to/output
```

**Use this toolbox when** you want to (a) match the exact methodology of the Zhang & Scaramuzza tutorial, (b) compare several algorithms across several datasets in one shot, or (c) generate the paper-ready boxplots and LaTeX tables that journal/conference reviewers expect. For everyday one-off checks, `evo` is faster to reach for.

---

#### TUM RGB-D `evaluate_ate.py` / `evaluate_rpe.py`

The original scripts from Sturm et al. Simpler than `evo`, but still in wide use for TUM RGB-D papers.

---


### 4.9 What to report when benchmarking a new VO/SLAM method

A minimal, defensible reporting recipe:

1. **Per sequence**: ATE-trans RMSE + ATE-trans median, with **explicit alignment** stated (e.g., "Sim(3), Umeyama"). Include rotation ATE if rotation accuracy is a claim.
2. **Aggregated across the dataset**: KITTI-style $t_{rel}$ (%) and $r_{rel}$ (deg/100 m), computed on the standard segment lengths for that dataset.
3. **A trajectory plot** with ground truth overlaid for at least one representative sequence.
4. **A boxplot of drift vs. segment length** (the sub-trajectory analysis of §4.6) for at least one sequence — this is what reveals whether the gain comes from better local accuracy or better global consistency.
5. **Failure cases**: number of times tracking was lost; how those frames are treated in the statistics.
6. **The exact `evo` or `rpg_trajectory_evaluation` command** used. Reproducibility is cheap; not reporting it is suspicious.

Numbers without these qualifiers are not meaningful for cross-paper comparison.

---


### 4.10 References

* Z. Zhang and D. Scaramuzza, *"A Tutorial on Quantitative Trajectory Evaluation for Visual(-Inertial) Odometry"*, **IROS 2018**. [[PDF]](https://rpg.ifi.uzh.ch/docs/IROS18_Zhang.pdf) · [[Toolbox]](https://github.com/uzh-rpg/rpg_trajectory_evaluation)
* J. Sturm, N. Engelhard, F. Endres, W. Burgard, D. Cremers, *"A Benchmark for the Evaluation of RGB-D SLAM Systems"*, **IROS 2012**. (Original ATE/RPE definitions used by the TUM RGB-D toolkit.)
* A. Geiger, P. Lenz, R. Urtasun, *"Are we ready for autonomous driving? The KITTI vision benchmark suite"*, **CVPR 2012**. (Origin of the sub-trajectory drift evaluation style.)
* S. Umeyama, *"Least-squares estimation of transformation parameters between two point patterns"*, **IEEE TPAMI 1991**. (Closed-form Sim(3) alignment.)
* M. Grupp, *"evo: Python package for the evaluation of odometry and SLAM"*, [github.com/MichaelGrupp/evo](https://github.com/MichaelGrupp/evo).

---
